# Topic Modeling with BERTopic — Parameter Search & Evaluation

Loads pre-computed embeddings for the **iGEM teams** and **SynBio papers** datasets,
runs a grid search over key UMAP/HDBSCAN parameters, evaluates each configuration
using **topic coherence** ($C_v$), **topic diversity**, and **DBCV**, picks the best
model per dataset, and saves the results.

In [44]:
import os
from itertools import product

import numpy as np
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)

In [45]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ASSETS_DIR     = os.path.join('..', 'assets')
EMBEDDINGS_DIR = os.path.join(ASSETS_DIR, 'embeddings')
MODELS_DIR     = os.path.join(ASSETS_DIR, 'topic_models')
os.makedirs(MODELS_DIR, exist_ok=True)

# ── PARAMETER GRID ────────────────────────────────────────────────────────────
PARAM_GRID_TEAMS = {
    'min_cluster_size': [8, 10, 15, 20],
    'umap_n_neighbors': [10, 15, 25],
    'umap_n_components': [5, 10],
}

PARAM_GRID_PAPERS = {
    'min_cluster_size': [20, 30, 40, 50],
    'umap_n_neighbors': [10, 15, 25],
    'umap_n_components': [5, 10],
}

## 1. Load embeddings and corpora

In [46]:
teams_embeddings  = np.load(os.path.join(EMBEDDINGS_DIR, 'teams_embeddings.npy'))
papers_embeddings = np.load(os.path.join(EMBEDDINGS_DIR, 'papers_embeddings.npy'))

teams_corpus  = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'teams_corpus.txt'), sep='\t')
papers_corpus = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'papers_corpus.txt'), sep='\t')

teams_docs  = teams_corpus['text'].tolist()
papers_docs = papers_corpus['text'].tolist()

print(f'Teams:  {teams_embeddings.shape[0]:,} docs, {teams_embeddings.shape[1]} dims')
print(f'Papers: {papers_embeddings.shape[0]:,} docs, {papers_embeddings.shape[1]} dims')

assert len(teams_docs) == teams_embeddings.shape[0]
assert len(papers_docs) == papers_embeddings.shape[0]

Teams:  4,548 docs, 384 dims
Papers: 24,202 docs, 384 dims


## 2. Helpers: model fitting and evaluation metrics

In [47]:
def fit_topic_model(
    docs: list[str],
    embeddings: np.ndarray,
    min_cluster_size: int = 15,
    umap_n_neighbors: int = 15,
    umap_n_components: int = 5,
) -> tuple[BERTopic, list[int], np.ndarray]:
    """Fit a BERTopic model using UMAP + HDBSCAN."""
    umap_model = UMAP(
        n_neighbors=umap_n_neighbors,
        n_components=umap_n_components,
        min_dist=0.0,
        metric='cosine',
        random_state=SEED,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=1,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
        gen_min_span_tree=True,
    )
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        min_topic_size=min_cluster_size,
        n_gram_range=(1, 3),
        language='english',
        calculate_probabilities=True,
        verbose=False,
    )
    topics, probs = topic_model.fit_transform(docs, embeddings)
    return topic_model, topics, probs

In [48]:
def coherence_cv(topic_model: BERTopic, docs: list[str], top_n: int = 10) -> float:
    """Compute C_v coherence over the top-N words of each non-outlier topic.

    BERTopic may produce multi-word n-grams (e.g. 'gene expression').
    We split those into individual tokens so every term is present in the
    unigram dictionary that Gensim builds from the corpus.
    """
    topics = topic_model.get_topics()
    topic_words = []
    for t in sorted(topics):
        if t == -1:
            continue
        words = []
        for word, _ in topics[t][:top_n]:
            words.extend(word.lower().split())  # split n-grams into unigrams
        topic_words.append(words)
    if not topic_words:
        return float('nan')
    tokenized = [doc.lower().split() for doc in docs]
    dictionary = Dictionary(tokenized)
    # Keep only tokens that exist in the dictionary
    topic_words = [
        [w for w in tw if w in dictionary.token2id]
        for tw in topic_words
    ]
    topic_words = [tw for tw in topic_words if len(tw) >= 2]
    if not topic_words:
        return float('nan')
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized,
        dictionary=dictionary,
        coherence='c_v',
    )
    return cm.get_coherence()


def topic_diversity(topic_model: BERTopic, top_n: int = 10) -> float:
    """Fraction of unique words across all topics' top-N word lists."""
    topics = topic_model.get_topics()
    all_words = [
        word
        for t in sorted(topics) if t != -1
        for word, _ in topics[t][:top_n]
    ]
    return len(set(all_words)) / len(all_words) if all_words else float('nan')


def dbcv_score(topic_model: BERTopic) -> float:
    """HDBSCAN relative validity (DBCV). Higher is better, range [-1, 1]."""
    return topic_model.hdbscan_model.relative_validity_

In [49]:
def grid_search(
    docs: list[str],
    embeddings: np.ndarray,
    param_grid: dict,
    label: str = '',
) -> pd.DataFrame:
    """Run all parameter combinations, evaluate each, return results DataFrame."""
    combos = list(product(
        param_grid['min_cluster_size'],
        param_grid['umap_n_neighbors'],
        param_grid['umap_n_components'],
    ))
    n_total = len(combos)
    rows = []
    best_score = -np.inf
    best_model_data = None

    for i, (mcs, nn, nc) in enumerate(combos, 1):
        print(f'[{label}] {i}/{n_total}  mcs={mcs}, nn={nn}, nc={nc} ... ', end='', flush=True)
        model, topics, probs = fit_topic_model(
            docs, embeddings,
            min_cluster_size=mcs, umap_n_neighbors=nn, umap_n_components=nc,
        )
        n_topics = model.get_topic_info().Topic.max() + 1
        outlier_frac = (np.array(topics) == -1).mean()
        cv  = coherence_cv(model, docs)
        div = topic_diversity(model)
        dbcv = dbcv_score(model)

        row = {
            'min_cluster_size': mcs,
            'n_neighbors': nn,
            'n_components': nc,
            'n_topics': n_topics,
            'outlier_frac': round(outlier_frac, 4),
            'coherence_cv': round(cv, 4),
            'diversity': round(div, 4),
            'dbcv': round(dbcv, 4),
        }
        rows.append(row)
        print(f'topics={n_topics}, C_v={cv:.4f}, div={div:.4f}, DBCV={dbcv:.4f}')

        # Track best model by C_v (primary), break ties by diversity
        if cv > best_score or (cv == best_score and div > (best_model_data or {}).get('diversity', -1)):
            best_score = cv
            best_model_data = {
                'model': model, 'topics': topics, 'probs': probs,
                'diversity': div, **row,
            }

    df = pd.DataFrame(rows).sort_values('coherence_cv', ascending=False).reset_index(drop=True)
    return df, best_model_data

## 3. Grid search: iGEM Teams

In [50]:
teams_results, teams_best = grid_search(
    teams_docs, teams_embeddings, PARAM_GRID_TEAMS, label='Teams'
)
teams_results

[Teams] 1/24  mcs=8, nn=10, nc=5 ... topics=197, C_v=0.5616, div=0.7416, DBCV=0.2163
[Teams] 2/24  mcs=8, nn=10, nc=10 ... topics=192, C_v=0.5585, div=0.7438, DBCV=0.2269
[Teams] 3/24  mcs=8, nn=15, nc=5 ... topics=178, C_v=0.5513, div=0.7449, DBCV=0.2148
[Teams] 4/24  mcs=8, nn=15, nc=10 ... topics=183, C_v=0.5676, div=0.7372, DBCV=0.2339
[Teams] 5/24  mcs=8, nn=25, nc=5 ... topics=161, C_v=0.5785, div=0.7342, DBCV=0.1823
[Teams] 6/24  mcs=8, nn=25, nc=10 ... topics=162, C_v=0.5689, div=0.7290, DBCV=0.1635
[Teams] 7/24  mcs=10, nn=10, nc=5 ... topics=158, C_v=0.5733, div=0.6949, DBCV=0.2130
[Teams] 8/24  mcs=10, nn=10, nc=10 ... topics=146, C_v=0.5648, div=0.6658, DBCV=0.2620
[Teams] 9/24  mcs=10, nn=15, nc=5 ... topics=147, C_v=0.5578, div=0.6986, DBCV=0.2185
[Teams] 10/24  mcs=10, nn=15, nc=10 ... topics=138, C_v=0.5579, div=0.6710, DBCV=0.2218
[Teams] 11/24  mcs=10, nn=25, nc=5 ... topics=134, C_v=0.5735, div=0.6701, DBCV=0.1628
[Teams] 12/24  mcs=10, nn=25, nc=10 ... topics=132, C

,min_cluster_size,n_neighbors,n_components,n_topics,outlier_frac,coherence_cv,diversity,dbcv
0,8,25,5,161,0.2408,0.5785,0.7342,0.1823
1,10,25,5,134,0.2485,0.5735,0.6701,0.1628
2,10,10,5,158,0.2170,0.5733,0.6949,0.2130
3,8,25,10,162,0.2348,0.5689,0.7290,0.1635
4,10,25,10,132,0.2348,0.5679,0.6538,0.1297
5,8,15,10,183,0.2183,0.5676,0.7372,0.2339
6,10,10,10,146,0.1887,0.5648,0.6658,0.2620
7,8,10,5,197,0.1911,0.5616,0.7416,0.2163
8,8,10,10,192,0.1887,0.5585,0.7438,0.2269
9,10,15,10,138,0.2102,0.5579,0.6710,0.2218


In [51]:
print('Best Teams configuration:')
print(f"  min_cluster_size = {teams_best['min_cluster_size']}")
print(f"  n_neighbors      = {teams_best['n_neighbors']}")
print(f"  n_components     = {teams_best['n_components']}")
print(f"  n_topics         = {teams_best['n_topics']}")
print(f"  C_v              = {teams_best['coherence_cv']}")
print(f"  diversity        = {teams_best['diversity']}")
print(f"  DBCV             = {teams_best['dbcv']}")
print(f"  outlier_frac     = {teams_best['outlier_frac']}")

Best Teams configuration:
  min_cluster_size = 8
  n_neighbors      = 25
  n_components     = 5
  n_topics         = 161
  C_v              = 0.5785
  diversity        = 0.7342
  DBCV             = 0.1823
  outlier_frac     = 0.2408


## 4. Grid search: SynBio Papers

In [52]:
papers_results, papers_best = grid_search(
    papers_docs, papers_embeddings, PARAM_GRID_PAPERS, label='Papers'
)
papers_results

[Papers] 1/24  mcs=20, nn=10, nc=5 ... topics=309, C_v=0.6561, div=0.6689, DBCV=0.1916
[Papers] 2/24  mcs=20, nn=10, nc=10 ... topics=322, C_v=0.6566, div=0.6752, DBCV=0.2191
[Papers] 3/24  mcs=20, nn=15, nc=5 ... topics=309, C_v=0.6606, div=0.6667, DBCV=0.1959
[Papers] 4/24  mcs=20, nn=15, nc=10 ... topics=299, C_v=0.6540, div=0.6719, DBCV=0.1902
[Papers] 5/24  mcs=20, nn=25, nc=5 ... topics=263, C_v=0.6626, div=0.6540, DBCV=0.1764
[Papers] 6/24  mcs=20, nn=25, nc=10 ... topics=275, C_v=0.6471, div=0.6600, DBCV=0.1796
[Papers] 7/24  mcs=30, nn=10, nc=5 ... topics=217, C_v=0.6401, div=0.6129, DBCV=0.1942
[Papers] 8/24  mcs=30, nn=10, nc=10 ... topics=205, C_v=0.6375, div=0.6263, DBCV=0.2097
[Papers] 9/24  mcs=30, nn=15, nc=5 ... topics=204, C_v=0.6451, div=0.6216, DBCV=0.1756
[Papers] 10/24  mcs=30, nn=15, nc=10 ... topics=202, C_v=0.6375, div=0.6243, DBCV=0.1732
[Papers] 11/24  mcs=30, nn=25, nc=5 ... topics=182, C_v=0.6380, div=0.6104, DBCV=0.1856
[Papers] 12/24  mcs=30, nn=25, nc=10

,min_cluster_size,n_neighbors,n_components,n_topics,outlier_frac,coherence_cv,diversity,dbcv
0,20,25,5,263,0.3726,0.6626,0.6540,0.1764
1,20,15,5,309,0.3312,0.6606,0.6667,0.1959
2,20,10,10,322,0.2957,0.6566,0.6752,0.2191
3,20,10,5,309,0.2718,0.6561,0.6689,0.1916
4,20,15,10,299,0.3211,0.6540,0.6719,0.1902
5,20,25,10,275,0.3471,0.6471,0.6600,0.1796
6,30,15,5,204,0.3081,0.6451,0.6216,0.1756
7,30,10,5,217,0.2902,0.6401,0.6129,0.1942
8,30,25,5,182,0.3629,0.6380,0.6104,0.1856
9,30,10,10,205,0.2987,0.6375,0.6263,0.2097


In [53]:
print('Best Papers configuration:')
print(f"  min_cluster_size = {papers_best['min_cluster_size']}")
print(f"  n_neighbors      = {papers_best['n_neighbors']}")
print(f"  n_components     = {papers_best['n_components']}")
print(f"  n_topics         = {papers_best['n_topics']}")
print(f"  C_v              = {papers_best['coherence_cv']}")
print(f"  diversity        = {papers_best['diversity']}")
print(f"  DBCV             = {papers_best['dbcv']}")
print(f"  outlier_frac     = {papers_best['outlier_frac']}")

Best Papers configuration:
  min_cluster_size = 20
  n_neighbors      = 25
  n_components     = 5
  n_topics         = 263
  C_v              = 0.6626
  diversity        = 0.654
  DBCV             = 0.1764
  outlier_frac     = 0.3726


## 5. Reduce outliers for iGEM teams

HDBSCAN assigns topic **-1** (noise) to documents that do not fall inside
any dense cluster.  While this is acceptable for the SynBio literature
(some papers may be genuinely off-topic), every iGEM team project is by
definition related to synthetic biology — its text may simply be too
short or idiosyncratic to land in a cluster.

BERTopic's `reduce_outliers` method reassigns noise documents to their
nearest topic based on embedding cosine similarity, without retraining
the model.  Setting `threshold=0` forces **all** outliers into a topic.

In [54]:
# ── Reassign all iGEM outlier documents to their nearest topic ─────────────
teams_model  = teams_best['model']
teams_topics = list(teams_best['topics'])

outliers_before = sum(1 for t in teams_topics if t == -1)

teams_topics = teams_model.reduce_outliers(
    teams_docs, teams_topics,
    strategy="embeddings",
    embeddings=teams_embeddings,
    threshold=0,
)
teams_model.update_topics(teams_docs, topics=teams_topics)

outliers_after = sum(1 for t in teams_topics if t == -1)
print(f'Teams outliers: {outliers_before:,} → {outliers_after:,}')

2026-03-26 23:26:41,706 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Teams outliers: 1,095 → 0


## 6. Save best models and outputs

In [55]:
# teams_model and teams_topics are already set (with outliers reduced)
papers_model  = papers_best['model']
papers_topics = papers_best['topics']

# ── Save models ───────────────────────────────────────────────────────────────
teams_model.save(os.path.join(MODELS_DIR, 'teams_topic_model'))
papers_model.save(os.path.join(MODELS_DIR, 'papers_topic_model'))

# ── Save topic info summaries ─────────────────────────────────────────────────
teams_info  = teams_model.get_topic_info()
papers_info = papers_model.get_topic_info()

teams_info.to_csv(os.path.join(MODELS_DIR, 'teams_topic_info.txt'), sep='\t', index=False)
papers_info.to_csv(os.path.join(MODELS_DIR, 'papers_topic_info.txt'), sep='\t', index=False)

# ── Save document-level topic assignments ─────────────────────────────────────
teams_corpus['topic'] = teams_topics
teams_corpus[['UT', 'topic']].to_csv(
    os.path.join(MODELS_DIR, 'teams_doc_topics.txt'), sep='\t', index=False
)
papers_corpus['topic'] = papers_topics
papers_corpus[['id', 'topic']].to_csv(
    os.path.join(MODELS_DIR, 'papers_doc_topics.txt'), sep='\t', index=False
)

# ── Save grid search results ─────────────────────────────────────────────────
teams_results.to_csv(os.path.join(MODELS_DIR, 'teams_grid_search.txt'), sep='\t', index=False)
papers_results.to_csv(os.path.join(MODELS_DIR, 'papers_grid_search.txt'), sep='\t', index=False)

print(f'All outputs saved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')

2026-03-26 23:26:53,459 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
2026-03-26 23:26:57,029 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


All outputs saved to ../assets/topic_models
  papers_doc_topics.txt
  papers_grid_search.txt
  papers_topic_info.txt
  papers_topic_model
  papers_topic_names.txt
  teams_doc_topics.txt
  teams_grid_search.txt
  teams_topic_info.txt
  teams_topic_model
  teams_topic_names.txt
